In [1]:
import polars as pl
from tqdm import tqdm

In [2]:
appv = pl.scan_parquet("/home/dnanexus/data_dir/appv_files/avg_pheno_per_var_quantitative_EUR_genebass1e6_PRScorr.parquet")

# Chain the filter and the much faster semi join
# appv = (
#     appv
#     .with_columns(
#         mean_pheno_value_rank = pl.col('mean_pheno_value').rank(method="average").over(["phenotype"]),
#         mean_pheno_value_count = pl.len().over(["phenotype"])
#     )
#     .with_columns(
#         mean_pheno_value_ptile = pl.col('mean_pheno_value_rank') / pl.col('mean_pheno_value_count')
#     )
#     .drop(['mean_pheno_value_count'])
# )

# appv.head().collect(engine='streaming')

In [3]:
unique_phenotypes = appv.select('phenotype').unique().collect(engine='streaming').to_series().sort()
unique_phenotypes

phenotype
str
"""age_at_death_int"""
"""age_at_hysterectomy_int"""
"""age_last_used_hormonereplaceme…"
"""age_started_hormonereplacement…"
"""age_started_wearing_glasses_or…"
…
"""weight_int"""
"""white_blood_cell_leukocyte_cou…"
"""whole_body_fat_mass_int"""


In [11]:
(
    appv
    .filter(pl.col('phenotype') == unique_phenotypes[0])
    .with_columns(
        mean_pheno_value_rank = pl.col('mean_pheno_value').rank(method="average").cast(pl.Float32)
    )
    .with_columns(
        mean_pheno_value_ptile = (pl.col('mean_pheno_value_rank') / pl.len()).cast(pl.Float32)
    )
    .collect(engine='streaming')
)

id,phenotype,n_individuals,mean_pheno_value,std_pheno_value,mean_pheno_value_rank,mean_pheno_value_ptile
str,str,u32,f32,f32,f32,f32
"""chr1:1366008:A:G""","""age_at_death_int""",6142,0.000776,0.527588,8.179128e6,0.492442
"""chr1:1365723:G:C""","""age_at_death_int""",14,-0.078654,0.688095,6.194531e6,0.372955
"""chr1:1360989:GACAC:G""","""age_at_death_int""",3443,0.007729,0.52576,8.598257e6,0.517677
"""chr1:1362470:AT:A""","""age_at_death_int""",3576,0.008641,0.525432,8.63319e6,0.51978
"""chr1:1363031:C:T""","""age_at_death_int""",3402,0.007385,0.526906,8.583263e6,0.516774
…,…,…,…,…,…,…
"""chr22:50578990:T:C""","""age_at_death_int""",1,0.453231,null,1.452058e7,0.874244
"""chr22:50579004:G:C""","""age_at_death_int""",1,-0.658198,null,1.148782e6,0.069165
"""chr22:50579009:T:A""","""age_at_death_int""",2,-0.359511,1.148125,2.938968e6,0.176947


In [4]:
output_dir = "/home/dnanexus/data_dir/appv_files/ptile_pheno_file"

for phenotype in tqdm(unique_phenotypes):
    (
        appv
        .filter(pl.col('phenotype') == phenotype)
        .with_columns(
            mean_pheno_value_rank=pl.col('mean_pheno_value')
                .rank(method="average")
                .cast(pl.Float32)
        )
        .with_columns(
            mean_pheno_value_ptile=(
                pl.col('mean_pheno_value_rank') / pl.len()
            ).cast(pl.Float32)
        )
        .sink_parquet(
            f"{output_dir}/avg_pheno_per_var_quantitative_EUR_genebass1e6_PRScorr_with_percentiles_{phenotype}.parquet",
            compression="zstd",  # Better compression
            row_group_size=100_000  # Optimize for reading
        )
    )

# Combine all files
(
    pl.scan_parquet(f"{output_dir}/*.parquet")
    .sink_parquet(
        "/home/dnanexus/data_dir/appv_files/avg_pheno_per_var_quantitative_EUR_genebass1e6_PRScorr_with_percentiles.parquet",
        compression="zstd"
    )
)

100%|██████████| 127/127 [19:58<00:00,  9.44s/it]
